# Markov Chains

**Seong Cho** · ACME Volume 2, Lab 5

**Lab objective:** construct, analyze, and interact with Markov chains (collections of states with specified probabilities for transitioning from one state to another, whose future behavior depends only on the current state), then apply a Markov chain to a natural language processing problem.

<sub>Lab specification from the *Volume 2* lab manual of [Foundations of Applied Mathematics](https://foundations-of-applied-mathematics.github.io/) (BYU ACME), © Dr. J. Humpherys, used under [CC BY 3.0 US](http://creativecommons.org/licenses/by/3.0/us/). Problem statements are quoted from the spec; the notes are condensed from it. Solutions are my own.</sub>

In [1]:
import numpy as np
from scipy import linalg as la

## State Space Models

Many systems can be described by a finite number of states. In a board game where players move by die rolls, each space is a state, and the probability of moving from one space to another depends only on the player's current location, not on where they were on previous turns.

A finite Markov chain has a *transition matrix* whose $(i,j)$th entry is the probability of moving from state $j$ to state $i$, so each of its columns sums to 1.

> **Note.** A transition matrix whose columns sum to 1 is called *column stochastic* (or left stochastic). The rows of a *row stochastic* (or right stochastic) transition matrix each sum to 1, and its $(i,j)$th entry is the probability of moving from state $i$ to state $j$. Both representations are common, but this lab uses column stochastic transition matrices exclusively.

In a simple weather model, the probability that tomorrow is hot given that today is hot is 0.7, and the probability that tomorrow is cold given that today is cold is 0.4. Assigning hot to the 0th row and column and cold to the 1st gives

$$\begin{array}{c c} & \begin{array}{c c} \text{hot today} & \text{cold today} \end{array} \\ \begin{array}{c} \text{hot tomorrow} \\ \text{cold tomorrow} \end{array} & \left[\begin{array}{c c} \quad 0.7 \quad & \quad 0.6 \quad \\ \quad 0.3 \quad & \quad 0.4 \quad \end{array}\right] \end{array}$$

The 0th column says that if it is hot today, there is a 70% chance that tomorrow is hot and a 30% chance that it is cold; the 1st column says that if it is cold today, there is a 60% chance of heat and a 40% chance of cold tomorrow. Markov chains can also be drawn as a *state diagram*, a directed graph whose nodes are the states and whose edges carry the transition probabilities (here: hot → hot 0.7, hot → cold 0.3, cold → hot 0.6, cold → cold 0.4).

## Problem 1: A random transition matrix

> **Problem 1.** Transition matrices for Markov chains are efficiently stored as NumPy arrays. Write a function that accepts an integer $n$ and returns the transition matrix for a random Markov chain with $n$ states.
>
> (Hint: use array broadcasting to avoid looping.)

In [2]:
def random_chain(n):
    """Create and return a transition matrix for a random Markov chain with
    'n' states. This should be stored as an nxn NumPy array.
    """
    A = np.random.random((n,n))
    y = [sum(A[:, i]) for i in range(n)]
    return A/y

In [3]:
np.random.seed(5)
A = random_chain(4)
print(np.round(A, 4))
print("column sums:", A.sum(axis=0))
for n in [1, 2, 10, 100]:
    An = random_chain(n)
    assert An.shape == (n, n) and np.all(An >= 0) and np.allclose(An.sum(axis=0), 1)

[[0.1533 0.4762 0.1069 0.375 ]
 [0.3372 0.3346 0.3962 0.2116]
 [0.2049 0.1027 0.0418 0.3015]
 [0.3047 0.0866 0.4551 0.1119]]
column sums: [1. 1. 1. 1.]


## Simulating State Transitions

A single draw from a binomial distribution with parameters $n$ and $p$ gives the number of successes out of $n$ independent experiments, each with probability $p$ of success, such as coin flips where $p$ is the probability of heads. NumPy's `random.binomial()` draws from it:

In [4]:
np.random.seed(2)
# Draw from the binomial distribution with n = 1 and p = .5 (flip 1 coin).
np.random.binomial(1, .5)       # 0 means the coin flip resulted in tails.

0

If today is hot, the "hot" column of the transition matrix is $[0.7, 0.3]^T$. To decide tomorrow's weather, draw from the binomial distribution with $n = 1$ and $p = 0.3$: a 1 (30% likelihood) means tomorrow is cold, and a 0 (70% likelihood) means it is hot. The spec's function:

```python
def forecast():
    """Forecast tomorrow's weather given that today is hot."""
    transition = np.array([[0.7, 0.6], [0.3, 0.4]])

    # Sample from a binomial distribution to choose a new state.
    return np.binomial(1, transition[1, 0])
```

(`np.binomial` here stands for `np.random.binomial`.)

## Problem 2: Simulating the two-state weather model

> **Problem 2.** Modify `forecast()` so that it accepts an integer parameter `days` and runs a simulation of the weather for the number of days given. Return a list containing the day-by-day weather predictions (0 for hot, 1 for cold). Assume the first day is hot, but do not include the data from the first day in the list of predictions. The resulting list should therefore have `days` entries.

In [5]:
def forecast(days):
    """Forecast tomorrow's weather given that today is hot."""
    transition = np.array([[0.7, 0.6], [0.3, 0.4]])

    # Sample from a binomial distribution to choose a new state.
    l = [0]
    for i in range(days):
        l.append(np.random.binomial(1, transition[1,l[i]]))
    return l[1:]

In [6]:
np.random.seed(2)
print(forecast(20))
assert len(forecast(20)) == 20 and set(forecast(1000)) == {0, 1}

[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 0, 1, 0]


## Larger Chains

Larger Markov chains need draws from a *multinomial distribution*, a multivariate generalization of the binomial distribution. A draw with parameters $n$ and $(p_1, p_2, \ldots, p_k)$ indicates which of $k$ outcomes occurs in $n$ different experiments; the classic example is a series of dice rolls, with 6 possible outcomes of equal probability.

In [7]:
np.random.seed(3)
die_probabilities = np.array([1./6, 1./6, 1./6, 1./6, 1./6, 1./6])
# Draw from the multinomial distribution with n = 1 (roll a single die).
np.random.multinomial(1, die_probabilities)     # The position of the 1 is the roll.

array([0, 0, 0, 0, 1, 0])

## Problem 3: A four-state weather model

> **Problem 3.** Let the following matrix be the transition matrix for a Markov chain modeling weather with four states: hot, mild, cold, and freezing.
>
> $$\begin{array}{c c} & \begin{array}{c c c c} \text{hot} & \text{mild} & \text{cold} & \text{freezing} \end{array} \\ \begin{array}{c} \text{hot} \\ \text{mild} \\ \text{cold} \\ \text{freezing} \end{array} & \left[\begin{array}{c c c c} 0.5 & 0.3 & 0.1 & 0 \\ 0.3 & 0.3 & 0.3 & 0.3 \\ 0.2 & 0.3 & 0.4 & 0.5 \\ 0 & 0.1 & 0.2 & 0.2 \end{array}\right] \end{array}$$
>
> Write a new function that accepts an integer parameter and runs the same kind of simulation as `forecast()`, but that uses this new four-state transition matrix. This time, assume that the first day is mild. Return a list containing the day-to-day results (0 for hot, 1 for mild, 2 for cold, and 3 for freezing).

In [8]:
def four_state_forecast(days):
    """Run a simulation for the weather over the specified number of days,
    with mild as the starting state, using the four-state Markov chain.
    Return a list containing the day-by-day results, not including the
    starting day.

    Examples:
        >>> four_state_forecast(3)
        [0, 1, 3]
        >>> four_state_forecast(5)
        [2, 1, 2, 1, 1]
    """
    transition = np.array([[0.5, 0.3, 0.1, 0],
                           [0.3, 0.3, 0.3, 0.3],
                           [0.2, 0.3, 0.4, 0.5],
                           [0, 0.1, 0.2, 0.2]])
    l = [1]
    for i in range(days):
        n = np.random.multinomial(1, transition[:,l[i]])
        l.append(np.where(n==1)[0][0])
    return l[1:]

Hot (0) never follows freezing (3) directly, and freezing never follows hot, since those transition probabilities are 0:

In [9]:
np.random.seed(3)
print([int(s) for s in four_state_forecast(20)])
sim = four_state_forecast(10000)
assert len(sim) == 10000 and set(sim) <= {0, 1, 2, 3}
pairs = set(zip([1] + sim[:-1], sim))
assert (0, 3) not in pairs and (3, 0) not in pairs

[1, 3, 1, 2, 2, 3, 2, 2, 1, 2, 2, 1, 0, 0, 0, 0, 1, 2, 2, 2]


## General State Distributions

For a Markov chain with $n$ states, a *state distribution vector* $\mathbf{x}$ is an $n \times 1$ vector with nonnegative entries summing to 1, where $x_i$ is the probability of being in state $i$. For the 2-state weather model, $\mathbf{x} = [0.8, 0.2]^T$ means an 80% chance that today is hot and a 20% chance that it is cold, and $\mathbf{x} = [0, 1]^T$ means that today is, with 100% certainty, cold.

If $A$ is an $n \times n$ transition matrix and $\mathbf{x}$ is a state distribution vector, then $A\mathbf{x}$ is also a state distribution vector: if $\mathbf{x}_k$ is the state distribution at time $k$, then $\mathbf{x}_{k+1} = A\mathbf{x}_k$ holds the probabilities of each state after one more transition. If there is an 80% chance that it will be hot 5 days from now, $\mathbf{x}_5 = [0.8, 0.2]^T$, then

In [10]:
A = np.array([[.7, .6], [.3, .4]])
x5 = np.array([.8, .2])
A @ x5          # x6: a 68% chance that 6 days from now will be hot.

array([0.68, 0.32])

## Convergent Transition Matrices

Given an initial state distribution $\mathbf{x}_0$, the relation $\mathbf{x}_{k+1} = A\mathbf{x}_k$ gives

$$\mathbf{x}_k = A\mathbf{x}_{k-1} = A(A\mathbf{x}_{k-2}) = A(A(A\mathbf{x}_{k-3})) = \cdots = A^k\mathbf{x}_0,$$

so the $(i,j)$th entry of $A^k$ is the probability of transition from state $j$ to state $i$ in $k$ steps. For the 2-state weather model, the entries of $A^k$ settle quickly:

In [11]:
for k in [1, 2, 3]:
    print(f"A^{k} =\n{np.linalg.matrix_power(A, k)}")

A^1 =
[[0.7 0.6]
 [0.3 0.4]]
A^2 =
[[0.67 0.66]
 [0.33 0.34]]
A^3 =
[[0.667 0.666]
 [0.333 0.334]]


As $k \to \infty$,

$$\lim_{k\to\infty} A^k = \begin{bmatrix} 2/3 & 2/3 \\ 1/3 & 1/3 \end{bmatrix}, \tag{5.1}$$

and for any initial state distribution $\mathbf{x}_0 = [a, b]^T$ with $a + b = 1$,

$$\lim_{k\to\infty}\mathbf{x}_k = \lim_{k\to\infty}A^k\mathbf{x}_0 = \begin{bmatrix} 2/3 & 2/3 \\ 1/3 & 1/3 \end{bmatrix}\begin{bmatrix} a \\ b \end{bmatrix} = \begin{bmatrix} 2(a+b)/3 \\ (a+b)/3 \end{bmatrix} = \begin{bmatrix} 2/3 \\ 1/3 \end{bmatrix}.$$

So $\mathbf{x}_k \to \mathbf{x} = [2/3, 1/3]^T$ regardless of the initial distribution: no matter the weather today, the probability that it is hot a week from now is approximately 66.67%, and approximately 2 out of 3 days in the year should be hot.

## Steady State Distributions

The distribution $\mathbf{x} = [2/3, 1/3]^T$ also satisfies

$$A\mathbf{x} = \begin{bmatrix} 7/10 & 3/5 \\ 3/10 & 2/5 \end{bmatrix}\begin{bmatrix} 2/3 \\ 1/3 \end{bmatrix} = \begin{bmatrix} 14/30 + 3/15 \\ 6/30 + 2/15 \end{bmatrix} = \begin{bmatrix} 2/3 \\ 1/3 \end{bmatrix} = \mathbf{x}.$$

Any $\mathbf{x}$ with $A\mathbf{x} = \mathbf{x}$ is a *steady state distribution* or *stable fixed point* of $A$; in other words, an eigenvector of $A$ with eigenvalue $\lambda = 1$. Every Markov chain has at least one steady state distribution. If some power $A^k$ of $A$ has all positive (nonzero) entries, then the steady state distribution is unique (a consequence of the Perron-Frobenius theorem, presented with spectral calculus in Volume I). In this case $\lim_{k\to\infty}A^k$ is the matrix whose columns all equal the unique steady state distribution, as in (5.1), and $\mathbf{x}$ can be found by iterating $\mathbf{x}_{k+1} = A\mathbf{x}_k$ from any state distribution vector $\mathbf{x}_0$.

> **Achtung!** Though every Markov chain has at least one steady state distribution, the procedure described above fails if $A^k$ fails to converge. For example,
>
> $$A = \begin{bmatrix} 0 & 0 & 1 \\ 0 & 1 & 0 \\ 1 & 0 & 0 \end{bmatrix}, \qquad A^k = \begin{cases} A & \text{if } k \text{ is odd} \\ I & \text{if } k \text{ is even,} \end{cases}$$
>
> so $A^k$ oscillates between two different matrices as $k \to \infty$. Furthermore, the steady state distribution is not always unique; this transition matrix, for example, has infinitely many.

## Problem 4: Steady state distributions

> **Problem 4.** Write a function that accepts an $n \times n$ transition matrix $A$, a convergence tolerance $\epsilon$, and a maximum number of iterations $N$. Generate a random state distribution vector $\mathbf{x}_0$ and calculate $\mathbf{x}_{k+1} = A\mathbf{x}_k$ until $\|\mathbf{x}_{k-1} - \mathbf{x}_k\| < \epsilon$. If $k$ exceeds $N$, raise a `ValueError` to indicate that $A^k$ does not converge. Return the approximate steady state distribution $\mathbf{x}$ of $A$.
>
> To test your function, use Problem 1 to generate a random transition matrix $A$. Verify that $A\mathbf{x} = \mathbf{x}$ and that the columns of $A^k$ approach $\mathbf{x}$ as $k \to \infty$. To compute $A^k$, use NumPy's (very efficient) algorithm for computing matrix powers.
>
> ```python
> >>> A = np.array([[.7, .6],[.3, .4]])
> >>> np.linalg.matrix_power(A, 10)       # Compute A^10.
> array([[ 0.66666667,  0.66666667],
>        [ 0.33333333,  0.33333333]])
> ```
>
> Finally, use your function to validate the results of Problems 2 and 3:
> 1. Calculate the steady state distributions corresponding to the transition matrices for each simulation.
> 2. Run each simulation for a large number of days and verify that the results match the steady state distribution (for example, check that approximately $2/3$ of the days are hot for the smaller weather model).

In [12]:
def steady_state(A, tol=1e-12, N=40):
    """Compute the steady state of the transition matrix A.

    Inputs:
        A ((n,n) ndarray): A column-stochastic transition matrix.
        tol (float): The convergence tolerance.
        N (int): The maximum number of iterations to compute.

    Raises:
        ValueError: if the iteration does not converge within N steps.

    Returns:
        x ((n,) ndarray): The steady state distribution vector of A.
    """
    x = np.random.random(len(A))
    x = x/sum(x)
    for k in range(N):
        y = x.copy()
        x = A@x
        if la.norm(x-y) < tol:
            break
    if la.norm(x-y) > tol:
        raise ValueError("A**k does not converge")
    return x

The spec's matrix power, then checks on a random chain from Problem 1: $A\mathbf{x} = \mathbf{x}$, the columns of $A^k$ approach $\mathbf{x}$, and $\mathbf{x}$ matches the eigenvector of $A$ for the eigenvalue 1 from `la.eig()`, scaled to sum to 1.

In [13]:
A = np.array([[.7, .6],[.3, .4]])
print(np.linalg.matrix_power(A, 10))        # Compute A^10.

def eig_steady_state(A):
    # Independent reference: the eigenvector for the eigenvalue closest to 1.
    vals, vecs = la.eig(A)
    v = np.real(vecs[:, np.argmin(np.abs(vals - 1))])
    return v / v.sum()

np.random.seed(4)
R = random_chain(6)
x = steady_state(R)
print("x       =", x)
print("A^50 (every column equals x):\n", np.linalg.matrix_power(R, 50))
assert np.allclose(R @ x, x) and np.isclose(x.sum(), 1)
assert np.allclose(np.linalg.matrix_power(R, 50), np.outer(x, np.ones(6)))
assert np.allclose(x, eig_steady_state(R))
for n in [2, 5, 20, 50]:
    Rn = random_chain(n)
    assert np.allclose(steady_state(Rn), eig_steady_state(Rn))

[[0.66666667 0.66666667]
 [0.33333333 0.33333333]]
x       = [0.22541339 0.14468541 0.1683562  0.20857091 0.10779357 0.14518052]
A^50 (every column equals x):
 [[0.22541339 0.22541339 0.22541339 0.22541339 0.22541339 0.22541339]
 [0.14468541 0.14468541 0.14468541 0.14468541 0.14468541 0.14468541]
 [0.1683562  0.1683562  0.1683562  0.1683562  0.1683562  0.1683562 ]
 [0.20857091 0.20857091 0.20857091 0.20857091 0.20857091 0.20857091]
 [0.10779357 0.10779357 0.10779357 0.10779357 0.10779357 0.10779357]
 [0.14518052 0.14518052 0.14518052 0.14518052 0.14518052 0.14518052]]


Validating Problems 2 and 3: the steady states of the two weather models, against the fraction of days spent in each state over 100,000 simulated days.

In [14]:
T2 = np.array([[0.7, 0.6], [0.3, 0.4]])
T4 = np.array([[0.5, 0.3, 0.1, 0],
               [0.3, 0.3, 0.3, 0.3],
               [0.2, 0.3, 0.4, 0.5],
               [0, 0.1, 0.2, 0.2]])

np.random.seed(6)
days = 100000
for name, T, sim in [("two-state ", T2, forecast), ("four-state", T4, four_state_forecast)]:
    x = steady_state(T)
    freq = np.bincount(sim(days), minlength=len(T)) / days
    print(f"{name}  steady state {np.round(x, 4)}   simulated {np.round(freq, 4)}")
    assert np.allclose(x, eig_steady_state(T))
    assert np.allclose(freq, x, atol=0.01)
assert np.allclose(steady_state(T2), [2/3, 1/3])

two-state   steady state [0.6667 0.3333]   simulated [0.669 0.331]


four-state  steady state [0.2466 0.3    0.3328 0.1207]   simulated [0.2491 0.2989 0.3318 0.1202]


The Achtung! example does not converge, so `steady_state()` raises a `ValueError`:

In [15]:
P = np.array([[0, 0, 1], [0, 1, 0], [1, 0, 0.]])
try:
    steady_state(P)
except ValueError as e:
    print("ValueError:", e)

ValueError: A**k does not converge


> **Note.** Problem 4 is a special case of the *power method*, an algorithm for calculating an eigenvector of a matrix corresponding to the eigenvalue of largest magnitude. The general version of the power method, together with a discussion of its convergence conditions, is discussed in another lab.

## Using Markov Chains to Simulate English

One of the original applications of Markov chains was to study natural languages. In the early 20th century, Markov used his chains to model how Russian switched from vowels to consonants, and by mid-century they had been used as an attempt to model English. Markov chains are, by themselves, insufficient to model very good English, but they can approach a fairly good model of bad English, with sometimes amusing results.

A Markov chain over English words is unaware of context or even of previous words in a sentence: if its current state is the word "continuous," the next word is more likely to be "function" than "raccoon," but even two likely words strung together may give gibberish. The subproblem here is to model the English of a specific file, the *training set*, so the transition probabilities reflect the sort of English its authors speak (a chain built from *The Complete Works of William Shakespeare* differs greatly from one built from a collection of academic journals).

### Making the Chain

English sentences are of varying length, so the model has two extra states: a start state (`$tart`) marking the beginning of a sentence and a stop state (`$top`) marking the end. A training set with $N$ unique words has an $(N+2) \times (N+2)$ transition matrix. The start state only transitions to words that begin a sentence in the training set, and only words that end a sentence transition to the stop state. The stop state is an *absorbing state*: once reached, the chain cannot transition back to another state.

Consider this small training set from Dr. Seuss (the file `test.txt` next to this notebook):

```text
I am Sam Sam I am.
Do you like green eggs and ham?
I do not like them, Sam I am.
I do not like green eggs and ham.
```

Including punctuation (so "ham?" and "ham." are distinct words) and keeping capitalization (so "Do" and "do" differ), there are 15 unique words:

```text
I  am  Sam  am.  Do  you  like  green  eggs  and  ham?  do  not  them,  ham.
```

With the start and stop states the transition matrix is $17 \times 17$. An easy way to index the states is in order of appearance: `$tart` is 0, `I` is 1, `am` is 2, `Sam` is 3, ..., `ham.` is 15, and `$top` is 16. The start state transitions to "I" and "Do", and "am.", "ham?", and "ham." each transition to the stop state. Counting the transitions and then dividing each column by its sum gives, in part (column = current state, row = next state),

| | `$tart` | I | am | Sam | ⋯ | ham. | `$top` |
|---|:-:|:-:|:-:|:-:|:-:|:-:|:-:|
| **`$tart`** | 0 | 0 | 0 | 0 | ⋯ | 0 | 0 |
| **I** | 3/4 | 0 | 0 | 2/3 | ⋯ | 0 | 0 |
| **am** | 0 | 1/5 | 0 | 0 | ⋯ | 0 | 0 |
| **Sam** | 0 | 0 | 1 | 1/3 | ⋯ | 0 | 0 |
| ⋮ | ⋮ | ⋮ | ⋮ | ⋮ | ⋱ | ⋮ | ⋮ |
| **ham.** | 0 | 0 | 0 | 0 | ⋯ | 0 | 0 |
| **`$top`** | 0 | 0 | 0 | 0 | ⋯ | 1 | 1 |

The 3/4 says that 3 out of 4 sentences in the training set start with "I"; the 2/3 and 1/3 say that "Sam" is followed by "I" twice and by "Sam" once. "am" (without a period) always transitions to "Sam", and "ham." (with a period) always transitions to the stop state. To avoid a column of zeros, a 1 goes in the bottom right corner, so the stop state always transitions to itself.

```text
Algorithm 5.1  Convert a training set of sentences into a Markov chain.
1: procedure MakeTransitionMatrix
2:     Count the number of unique words in the training set.
3:     Initialize a square array of zeros of the appropriate size to be the transition
       matrix (remember to account for the start and stop states).
4:     Initialize a list of states, beginning with "$tart".
5:     for each sentence in the training set do
6:         Split the sentence into a list of words.
7:         Add each new word in the sentence to the list of states.
8:         Convert the list of words into a list of indices indicating which row and
           column of the transition matrix each word corresponds to.
9:         Add 1 to the entry of the transition matrix corresponding to
           transitioning from the start state to the first word of the sentence.
10:        for each consecutive pair (x, y) of words in the list of words do
11:            Add 1 to the entry of the transition matrix corresponding to
               transitioning from state x to state y.
12:        Add 1 to the entry of the transition matrix corresponding to
           transitioning from the last word of the sentence to the stop state.
13:    Make sure the stop state transitions to itself.
14:    Normalize each column by dividing by the column sums.
```

## Problems 5 and 6: The `SentenceGenerator` class

> **Problem 5.** Write a class called `SentenceGenerator`. The constructor should accept a filename (the training set). Read the file and build a transition matrix from its contents as described in Algorithm 5.1.
>
> You may assume that the file has one complete sentence written on each line, and your implementation may be either column- or row-stochastic.

> **Problem 6.** Add a method to the `SentenceGenerator` class called `babble()`. Begin at the start state and use the strategy from Problem 3 to repeatedly transition through the object's Markov chain. Keep track of the path through the chain and the corresponding sequence of words. When the stop state is reached, stop transitioning to terminate the simulation. Return the resulting sentence as a single string.
>
> For example, your `SentenceGenerator` class should be able to create random sentences that sound somewhat like Yoda speaking.
>
> ```python
> >>> yoda = SentenceGenerator("yoda.txt")
> >>> for _ in range(3):
> ...     print(yoda.babble())
> ...
> Impossible to my size, do not!
> For eight hundred years old to enter the dark side of Congress there is.
> But beware of the Wookiees, I have.
> ```

My class keeps the words in `self.words` (index $i$ in the list is state $i + 1$) and builds a column-stochastic `self.transition`, with the start state first and the stop state last.

In [16]:
class SentenceGenerator(object):
    """Markov chain creator for simulating bad English.

    Attributes:
        (what attributes do you need to keep track of?)

    Example:
        >>> yoda = SentenceGenerator("Yoda.txt")
        >>> print(yoda.babble())
        The dark side of loss is a path as one with you.
    """
    def __init__(self, filename):
        """Read the specified file and build a transition matrix from its
        contents. You may assume that the file has one complete sentence
        written on each line.
        """
        sentences = []
        with open(filename, 'r') as file:
            sentences = file.read().split('\n')
        if len(sentences[-1]) == 0:
            sentences = sentences[:-1]
        self.words = []
        for i in range(len(sentences)):
            self.words += sentences[i].split(' ')
        self.words = list(set(self.words))
        count = len(self.words)
        self.transition = np.zeros((count+2, count+2))
        for i in range(len(sentences)):
            s = sentences[i].split(' ')
            self.transition[self.words.index(s[0])+1, 0] += 1
            for j in range(len(s)-1):
                self.transition[self.words.index(s[j+1])+1, self.words.index(s[j])+1] += 1
            self.transition[-1, self.words.index(s[-1])+1] += 1
        self.transition[-1,-1] += 1.
        y = np.array([sum(self.transition[:, i]) for i in range(len(self.transition))])
        y[y==0.0] = 1
        self.transition = self.transition/y

    def babble(self):
        """Begin at the start sate and use the strategy from
        four_state_forecast() to transition through the Markov chain.
        Keep track of the path through the chain and the corresponding words.
        When the stop state is reached, stop transitioning and terminate the
        sentence. Return the resulting sentence as a single string.
        """
        l = [0]
        i = 0
        while i == 0:
            n = np.random.multinomial(1, self.transition[:,l[-1]])
            l.append(np.where(n==1)[0][0])
            if n[-1] == 1:
                break
        s = [self.words[i-1] for i in l[1:-1]]
        return ' '.join(s)

Checks on the Dr. Seuss training set against the counts worked out above. The words come from a `set`, so their order (and therefore the state indices) depends on Python's string hashing; the checks look states up by word.

In [17]:
seuss = SentenceGenerator("test.txt")
T = seuss.transition
state = lambda w: seuss.words.index(w) + 1          # $tart is 0, $top is -1.
print(len(seuss.words), "unique words:", sorted(seuss.words))
print("transition matrix", T.shape)

assert len(seuss.words) == 15 and T.shape == (17, 17)
assert np.allclose(T.sum(axis=0), 1)
assert np.isclose(T[state("I"), 0], 3/4) and np.isclose(T[state("Do"), 0], 1/4)
assert np.isclose(T[state("am"), state("I")], 1/5)
assert np.isclose(T[state("Sam"), state("am")], 1)
assert np.isclose(T[state("I"), state("Sam")], 2/3) and np.isclose(T[state("Sam"), state("Sam")], 1/3)
for w in ["am.", "ham?", "ham."]:
    assert np.isclose(T[-1, state(w)], 1)
assert T[-1, -1] == 1 and np.allclose(T[:-1, -1], 0)     # $top is absorbing.
assert np.allclose(T[0, :], 0)                            # Nothing returns to $tart.

15 unique words: ['Do', 'I', 'Sam', 'am', 'am.', 'and', 'do', 'eggs', 'green', 'ham.', 'ham?', 'like', 'not', 'them,', 'you']
transition matrix (17, 17)


Babbling. The saved output was produced with `PYTHONHASHSEED=0` as well as the NumPy seed, since the state order depends on both. Every consecutive pair of words in a generated sentence appears somewhere in the training set.

In [18]:
np.random.seed(7)
for _ in range(4):
    print(seuss.babble())

yoda = SentenceGenerator("yoda.txt")
print(f"\nyoda.txt: {len(yoda.words)} unique words, transition matrix {yoda.transition.shape}\n")
for _ in range(6):
    print(yoda.babble())

with open("yoda.txt") as f:
    lines = [line.rstrip('\n').split(' ') for line in f if line.strip()]
pairs = {(a, b) for s in lines for a, b in zip(s, s[1:])}
assert np.allclose(yoda.transition.sum(axis=0), 1)
for _ in range(200):
    s = yoda.babble().split(' ')
    assert all(p in pairs for p in zip(s, s[1:]))

I am.
Do you like green eggs and ham.
I am.
I am Sam I am.



yoda.txt: 698 unique words, transition matrix (700, 700)

If so big eating food of the council we will I am sure of: do you that misread could have Qui-Gon.
A Jedi will.
There is in this puzzle there is, and too much about yourself you no more, no time it is.
Luke, when gone am sure of: do not!
That is your rule is, easier for attack.
Its energy surrounds us to begin the Force.


## Additional Material

### Large training sets

The approach in Problems 5 and 6 begins to fail as the training set grows: a single Shakespearean play may not cause memory problems, but *The Complete Works of William Shakespeare* certainly will. For larger data sets, build the transition matrix as a `scipy.sparse.lil_matrix` (easy to build incrementally), then convert it to `csc_matrix` format (fast column operations). How are the resulting sentences different if a very large training set is used instead of a small one?

### Variations on the English model

Choosing a different state space produces different results. A few possibilities:

- Give each punctuation mark its own state: instead of "ham?" and "ham.", use three states "ham", "?", and ".", with "ham" transitioning to both punctuation states.
- Model paragraphs instead of sentences: add a `$tartParagraph` state that always transitions to `$tartSentence`, and a `$topParagraph` state that is sometimes transitioned to from `$topSentence`.
- Let the states be individual letters instead of words, with a state for the spaces between words (explored further in Volume III with hidden Markov models).
- Make the next state depend on both the current and previous states, a *Markov chain of order 2*, so every three consecutive words in a generated sentence come from the training set.
- Simulate a random conversation between $n$ people: build a chain $M_i$ for each person and a chain $C$ whose states are the $M_i$, describing the conversation transitions from person to person. Generate a sentence from $M_1$, then use $C$ to pick the next speaker, and so on.

### Natural language processing tools

The model of Problems 5 and 6 is a natural language processing application. Python's `nltk` module (natural language toolkit, not part of the standard library; see http://www.nltk.org/) has many tools for such problems; for example, `nltk.sent_tokenize()` splits a single string into sentences.

```python
>>> from nltk import sent_tokenize
>>> with open("yoda.txt", 'r') as yoda:
...     sentences = sent_tokenize(yoda.read())
...
>>> print(sentences)
['Away with your weapon!',
 'I mean you no harm.',
 'I am wondering - why are you here?',
...
```